In [ ]:
!python3 --version 

Python 3.13.7


In [2]:
!pip3 install transformers sentence_transformers faiss-cpu fastapi uvicorn pyngrok tf-keras

  Using cached transformers-4.56.2-py3-none-any.whl.metadata (40 kB)
  Using cached sentence_transformers-5.1.1-py3-none-any.whl.metadata (16 kB)
  Using cached faiss_cpu-1.12.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.1 kB)
  Using cached fastapi-0.118.0-py3-none-any.whl.metadata (28 kB)
  Using cached uvicorn-0.37.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached pyngrok-7.4.0-py3-none-any.whl.metadata (8.1 kB)
  Using cached tf_keras-2.20.1-py3-none-any.whl.metadata (1.8 kB)
  Using cached filelock-3.19.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached huggingface_hub-0.35.3-py3-none-any.whl.metadata (14 kB)
  Using cached regex-2025.9.18-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.8 kB)
  Using cached safetensors-0.6.2-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
 

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

# Descargar y guardar el modelo de embeddings localmente
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embedder.save("modelos/all-MiniLM-L6-v2")

# Descargar y guardar el tokenizer y modelo de lenguaje x
modelo_id = "microsoft/Phi-4-mini-instruct"
tokenizer = AutoTokenizer.from_pretrained(modelo_id)
tokenizer.save_pretrained("modelos/Phi-4-mini-instruct")

modelo = AutoModelForCausalLM.from_pretrained(modelo_id)
modelo.save_pretrained("modelos/Phi-4-mini-instruct")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [3]:
## Correr desde aqui....!!
print('Hola')

Hola


In [4]:
# Cargar tokenizer y modelo de lenguaje localmente
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
#torch.set_num_threads(8)

tokenizer = AutoTokenizer.from_pretrained("../modelos/Phi-4-mini-instruct")
modelo = AutoModelForCausalLM.from_pretrained("../modelos/Phi-4-mini-instruct", dtype=torch.bfloat16).to("cuda")
#tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-4-mini-instruct")
#modelo = AutoModelForCausalLM.from_pretrained("microsoft/Phi-4-mini-instruct", dtype=torch.bfloat16).to("cuda")


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [5]:
torch.get_num_threads()

16

In [5]:
from sentence_transformers import SentenceTransformer

# Cargar modelo localmente
embedder = SentenceTransformer("../modelos/all-MiniLM-L6-v2")

In [6]:
import faiss
import numpy as np

embeddings = np.load("../embeddings/CHUNK_PRODUCTO.npy")
#embeddings = np.concatenate((embeddings,np.load("embeddings/CHUNK_FICHA.npy")))
#embeddings = np.concatenate((embeddings,np.load("embeddings/CHUNK_DESCRIPCION.npy")))
#embeddings = np.concatenate((embeddings,np.load("embeddings/CHUNK_CARACTERISTICAS.npy")))
#embeddings = np.concatenate((embeddings,np.load("embeddings/CHUNK_OBSERVACIONES.npy")))
#embeddings = np.concatenate((embeddings,np.load("embeddings/CHUNK_RECOMENDACIONES.npy")))

# Crear índice FAISS
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

In [7]:
import pandas as pd

df = pd.read_csv("../data/processed/productos_corpus.csv",delimiter=",")
df['CHUNK_DESCRIPCION'] = df['CHUNK_DESCRIPCION'].fillna('')
df['CHUNK_CARACTERISTICAS'] = df['CHUNK_CARACTERISTICAS'].fillna('')
df['CHUNK_OBSERVACIONES'] = df['CHUNK_OBSERVACIONES'].fillna('')
df['CHUNK_RECOMENDACIONES'] = df['CHUNK_RECOMENDACIONES'].fillna('')
tamanio = len(df)

def getChunk(n):
    cociente, resto = divmod(n, tamanio)
    chunk = ['CHUNK_PRODUCTO', 'CHUNK_FICHA', 'CHUNK_DESCRIPCION', 'CHUNK_CARACTERISTICAS', 'CHUNK_OBSERVACIONES', 'CHUNK_RECOMENDACIONES']
    return str(df.iloc[resto][chunk[cociente]])

In [8]:
getChunk(0)

'ovalin circular para sobreponer vidriotransparente de la marca orange del area de baños para los sanitarios de la linea ovalines de procedencia importado'

In [12]:
def modelScore(pregunta):
    contexto = "ovalin circular para sobreponer vidrio transparente de la marca orange del area de baños para los sanitarios de la linea ovalines de procedencia importado al precio de 79.9 soles"
    prompt = f"""
    CONTEXTO:
    {contexto}
    PREGUNTA: {pregunta}
    RESPUESTA:"""
    inputs = tokenizer(prompt, return_tensors="pt").to(modelo.device)
    outputs = modelo.generate(**inputs, min_new_tokens=10, max_new_tokens=25, do_sample=True, top_p=0.5, temperature=0.1)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [13]:
modelScore('Cual es la procedencia del ovalin?')

'\n    CONTEXTO:\n    ovalin circular para sobreponer vidrio transparente de la marca orange del area de baños para los sanitarios de la linea ovalines de procedencia importado al precio de 79.9 soles\n    PREGUNTA: Cual es la procedencia del ovalin?\n    RESPUESTA: La procedencia del ovalin es de procedencia importado.\n    CONTEXTO:\n    ovalin circular para sobreponer'

In [14]:
modelScore('Cual es el precio de ovalin?')

'\n    CONTEXTO:\n    ovalin circular para sobreponer vidrio transparente de la marca orange del area de baños para los sanitarios de la linea ovalines de procedencia importado al precio de 79.9 soles\n    PREGUNTA: Cual es el precio de ovalin?\n    RESPUESTA: El precio de ovalin es de 79.9 soles.\n    CONTEXTO:\n    ovalin circular para sobreponer'

In [15]:
modelScore('el ovalin es circular o cuadrado?')

'\n    CONTEXTO:\n    ovalin circular para sobreponer vidrio transparente de la marca orange del area de baños para los sanitarios de la linea ovalines de procedencia importado al precio de 79.9 soles\n    PREGUNTA: el ovalin es circular o cuadrado?\n    RESPUESTA: el ovalin es circular\n    CONTEXTO:\n    ovalin circular para sobreponer vidrio transparente de la marca orange del'

In [16]:
modelScore('de que marca es el ovalin?')

'\n    CONTEXTO:\n    ovalin circular para sobreponer vidrio transparente de la marca orange del area de baños para los sanitarios de la linea ovalines de procedencia importado al precio de 79.9 soles\n    PREGUNTA: de que marca es el ovalin?\n    RESPUESTA: el ovalin es de la marca orange\n    CONTEXTO:\n    ovalin circular para sobreponer vidrio transparente de la'

In [17]:
modelScore('de que area es el ovalin?')

'\n    CONTEXTO:\n    ovalin circular para sobreponer vidrio transparente de la marca orange del area de baños para los sanitarios de la linea ovalines de procedencia importado al precio de 79.9 soles\n    PREGUNTA: de que area es el ovalin?\n    RESPUESTA: el ovalin es un tipo de vidrio de origen importado, utilizado para cubrir los sanitarios en la línea ovalines,'

In [9]:
def modelx(pregunta, k=3, temperatura=0.2):
    #print('1.- encode')
    # Embed la pregunta
    pregunta_emb = embedder.encode([pregunta], convert_to_numpy=True)

    # Buscar los k textos más cercanos
    distancias, indices = index.search(pregunta_emb, k)
    contexto = "\n".join([getChunk(i) for i in indices[0]])

    #print(contexto)
    #print(distancias)

    prompt = f"""
    CONTEXTO:
    {contexto}
    PREGUNTA: {pregunta}
    RESPUESTA:"""

   

    inputs = tokenizer(prompt, return_tensors="pt").to(modelo.device)
    #print('3.- generate')
    outputs = modelo.generate(**inputs, min_new_tokens=10, max_new_tokens=250, do_sample=True, top_p=0.5, temperature=temperatura)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [10]:
import re
def respuestaModel(pregunta,k=3,temperatura=0.2):
    tok = modelx(pregunta, k,temperatura)
    # Buscar la primera respuesta
    #match = re.search(r"RESPUESTA:\s*(.+?)(?=\n\s*PREGUNTA:|\Z)", tok, re.DOTALL | re.IGNORECASE)
    #return match.group(1).strip()

    respuestas = re.findall(r"RESPUESTA:\s*(.+?)(?=\n\s*PREGUNTA:|\n\s*CONTEXTO:|\Z)", tok, re.DOTALL | re.IGNORECASE)

    # Filtramos respuestas que al menos terminan en un número (como "179.9 soles")
    #respuestas_validas = [r.strip() for r in respuestas if re.search(r"\d+(\.\d+)?\s+soles", r)]
    return respuestas[0].strip()

In [20]:
print(modelx('quiero un ovalin para mi baño, cuales tienen?',3))


    CONTEXTO:
    ovalin circular priano negro mate de la marca trebol del area de baños para los sanitarios de la linea ovalines de procedencia nacional
ovalin para sobreponer luna blanco de la marca trebol del area de baños para los sanitarios de la linea ovalines de procedencia nacional
ovalin para sobreponer divani blanco de la marca trebol del area de baños para los sanitarios de la linea ovalines de procedencia nacional
    PREGUNTA: quiero un ovalin para mi baño, cuales tienen?
    RESPUESTA: ovalin para sobreponer luna blanco de la marca trebol del area de baños para los sanitarios de la linea ovalines de procedencia nacional
    CONTEXTO:
    ovalin circular priano negro mate de la marca trebol del area de baños para los sanitarios de la linea ovalines de procedencia nacional
ovalin para sobreponer luna blanco de la marca trebol del area de baños para los sanitarios de la linea ovalines de procedencia nacional
ovalin para sobreponer divani blanco de la marca trebol del area d

In [21]:
print(respuestaModel('quiero un ovalin para mi baño, cuales tienen?',3))

ovalin para sobreponer luna blanco de la marca trebol del area de baños para los sanitarios de la linea ovalines de procedencia nacional


In [22]:
print(respuestaModel('quiero una cama, cual me recomiendas?',3))

te recomiendo la marca "marmol" es muy buena, te recomiendo la marca "marmol" es muy buena, te recomiendo la marca "marmol" es muy buena, te recomiendo la marca "marmol" es muy buena, te recomiendo la marca "marmol" es muy buena, te recomiendo la marca "marmol" es muy buena, te recomiendo la marca "marmol" es muy buena, te recomiendo la marca "marmol" es muy buena, te recomiendo la marca "marmol" es muy buena, te recomiendo la marca "marmol" es muy buena, te recomiendo la marca "marmol" es muy buena, te recomiendo la marca "marmol" es muy buena, te recomiendo la marca "marmol" es muy buena, te recomiendo la marca "marmol" es muy buena, te recomiendo la marca "marmol" es muy buena, te recomiendo la marca "marmol" es muy buena, te recomiendo la marca "marmol" es muy buena, te recomiendo la marca "marmol" es muy buena, te recomiendo la marca "marmol" es muy buena, te recomiendo la


In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
messagesX = [
    {"role": "system", "content": "Hola, en que puedo ayudarte?"},
    {"role": "user", "content": "Soy Andree, quiero saber cuantos dias tiene un año"},
    {"role": "assistant", "content": "un año tiene 365 dias"},
    {"role": "user", "content": "y cuantos viernes? ademas recuerdas mi nombre?"},
]

messagesY = [
    {"role": "system", "content": "Hola, en que puedo ayudarte?"},
    {"role": "user", "content": "contexto: el ovalin para sobreponer divani blanco de la marca trebol del area de baños para los sanitarios de la linea ovalines de procedencia nacional al precio de 179.9 soles"},
    #{"role": "assistant", "content": "un año tiene 365 dias"},
    {"role": "user", "content": "cuanto cuesta el ovalin? divani"},
    {"role": "assistant", "content": "El precio del Ovalin mencionado es de 179.9 soles. Sin embargo, es importante tener en cuenta que el Ovalin es un producto de decoración, específicamente un tipo de alfombra, y no un divani. Si estás buscando un divani, es posible que necesites buscar opciones separadas, ya que los precios y las características pueden variar significativamente."},
    {"role": "user", "content": "cual es la marca de ese ovalin?"}
]

messages = [ 
    {"role": "system", "content": "contexto: el ovalin para sobreponer divani blanco de la marca trebol del area de baños para los sanitarios de la linea ovalines de procedencia nacional al precio de 179.9 soles"},
    {"role": "user", "content": "cuanto cuesta el ovalin divani?"},
    {"role": "assistant", "content": "El ovalin divani cuesta 179.9 soles. Este producto es un divani blanco de la marca Trebol, diseñado para el área de baños de los sanitarios, y su procedencia es nacional."},
    {"role": "user", "content": "cual es la marca?"}
]
 
pipe = pipeline(
    "text-generation",
    model=modelo,
    tokenizer=tokenizer,
)
 
generation_args = {
    "max_new_tokens": 500,
    "return_full_text": False,
    "temperature": 0.0,
    "do_sample": False,
}
 
output = pipe(messages, **generation_args)
print(output[0]['generated_text'])


Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


La marca del ovalin divani es Trebol.


In [12]:
def promartBoot(texto,mensajes,k):
    # --------
    pregunta_emb = embedder.encode([texto], convert_to_numpy=True)
    # Buscar los k textos más cercanos
    distancias, indices = index.search(pregunta_emb, k)
    contexto = "\n".join([getChunk(i) for i in indices[0]])
    mensajes.append({"role": "system","content":"CONTEXTO: "+contexto})
    #print(contexto)
    print('----------------')
    # --------
    
    mensajes.append({"role": "user","content":texto})
    
    pipe = pipeline("text-generation", model=modelo, tokenizer=tokenizer)
     
    generation_args = {
        "max_new_tokens": 500,
        "return_full_text": False,
        "temperature": 0.0,
        "do_sample": False,
    }
     
    output = pipe(mensajes, **generation_args)
    return output[0]['generated_text']    

In [13]:
promartBoot('necesito un ovalin circular mimbell',[],3)

Device set to use cuda:0


----------------


'Parece que estás buscando un Ovalin circular mimbell, que es un tipo de tapicería de alta calidad. Los Ovalins son conocidos por su diseño elegante y su uso en interiores de lujo, como baños de lujo, pero también pueden encontrarse en otras áreas de la casa.\n\nSi estás buscando comprar un Ovalin circular mimbell, puedes visitar tiendas de tapicería, tiendas especializadas en muebles o buscar en línea en tiendas de productos de lujo. Marcas como Ovalin, Vainsa y Centurion, que mencionaste anteriormente, suelen ofrecer productos de alta calidad, pero es importante verificar si ofrecen tapicería mimbel o si puedes encontrar un proveedor que pueda hacerla para ti.\n\nSi estás buscando un producto específico de una marca, te recomendaría contactar directamente con la marca o con un proveedor autorizado para saber si pueden ofrecerte el producto que buscas.'

In [14]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import nest_asyncio
import uvicorn
from pyngrok import ngrok

In [16]:
nest_asyncio.apply()
app = FastAPI()

app.add_middleware(
        CORSMiddleware,
        allow_origins=["*"],  # Your defined list of allowed origins
        allow_credentials=True,  # Allow cookies and authorization headers
        allow_methods=["*"],  # Allow all HTTP methods (GET, POST, PUT, etc.)
        allow_headers=["*"],  # Allow all headers
    )

class Body(BaseModel):
    pregunta: str
    k: int
    history: list

@app.post("/clasificar")
def clasificar(body: Body):
    texto = body.pregunta
    k = body.k
    history = body.history
    return {"respuesta": promartBoot(texto,history,k)}

# Inicia el tunel ngrok
#ngrok.set_auth_token("1o4ItumAvMRlaYTG9dxyHKNInZq_48f9Txagc76SkaUUpoZB4")
#public_url = ngrok.connect(8000)
#print(f"🔗 API disponible en: {public_url}")


#uvicorn.run(app, port=8000)

In [17]:
# Inicia el tunel ngrok
ngrok.set_auth_token("1o4ItumAvMRlaYTG9dxyHKNInZq_48f9Txagc76SkaUUpoZB4")
public_url = ngrok.connect(8000)
print(f"🔗 API disponible en: {public_url}")

🔗 API disponible en: NgrokTunnel: "https://5723676c6e68.ngrok-free.app" -> "http://localhost:8000"


In [18]:
import threading
import uvicorn

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run, daemon=True).start()

INFO:     Started server process [47]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2001:1388:6660:2a23:8068:a983:f62e:68af:0 - "OPTIONS /clasificar HTTP/1.1" 200 OK


Device set to use cuda:0


----------------
INFO:     2001:1388:6660:2a23:8068:a983:f62e:68af:0 - "POST /clasificar HTTP/1.1" 200 OK


Device set to use cuda:0


...

----------------
INFO:     2001:1388:6660:2a23:8068:a983:f62e:68af:0 - "POST /clasificar HTTP/1.1" 200 OK


Device set to use cuda:0


----------------
INFO:     2001:1388:6660:2a23:8068:a983:f62e:68af:0 - "POST /clasificar HTTP/1.1" 200 OK


Device set to use cuda:0


----------------


In [28]:
print('holax')

holax


In [47]:
import uvicorn
from fastapi import FastAPI
import nest_asyncio

nest_asyncio.apply()

app = FastAPI()

@app.get("/")
async def read_root():
    return {"message": "Hello from FastAPI in Jupyter!"}

@app.get("/items/{item_id}")
async def read_item(item_id: int):
    return {"item_id": item_id, "data": f"Item {item_id} details"}

import threading

def run_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_uvicorn)
thread.start()

INFO:     Started server process [704]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [53]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI

nest_asyncio.apply()   # <-- Esto permite reusar el loop de Jupyter

app = FastAPI()

@app.get("/")
def read_root():
    return {"mensaje": "Hola desde FastAPI en Jupyter 🚀"}

@app.get("/hola/{nombre}")
def saludar(nombre: str):
    return {"saludo": f"Hola {nombre}!"}

uvicorn.run(app, host="0.0.0.0", port=8000)

RuntimeError: asyncio.run() cannot be called from a running event loop

In [56]:
import threading
import uvicorn

def run():
    uvicorn.run(app, host="0.0.0.0", port=8002)

threading.Thread(target=run, daemon=True).start()

INFO:     Started server process [704]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8002 (Press CTRL+C to quit)
